# GenPTW: four inpainting methods

Repository: [https://github.com/GanZhenliang/GenPTW](https://github.com/GanZhenliang/GenPTW)

Paper: [https://doi.org/10.1609/aaai.v40i5.37412](https://doi.org/10.1609/aaai.v40i5.37412)

Inspected commit: `8c22fc04dcf9d896eaeabd034d09918588e82cb3`

Watermark stage: `post_hoc`
Domain: image (framewise video adapter)

This method can be applied to existing media, so the notebook uses the fixed MOSE sample.

ProPainter, DiffuEraser, EffectErase, and E2FGVI-HQ receive the same authentic watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for that pipeline, then both reported outputs are trimmed to the authentic length. EffectErase applies its released Wan 4n+1 temporal adapter; E2FGVI-HQ uses the authors' official arbitrary-resolution test entry point.

The paper/released training code supports post-generation embedding by VAE-encoding existing COCO images and decoding the same latent with `decode_wm`. This notebook applies that path framewise to MOSE. It records the 512 x 512 center-crop/resize adapter and uses GenPTW's native 64-bit decoder and localizer after both removals.


## Authenticity contract

- Watermark preparation: Use the released COCO evaluation path: VAE-encode each aligned existing frame and decode the same latent through decode_wm with one fixed 64-bit message.
- Checkpoint status: REQUIRES msg_decoder.pth, localizer.pth, and diffusion_pytorch_model.safetensors from the authors' checkpoint bundle, plus the Stable Diffusion 2 base components.
- Protocol note: GenPTW is image-native but its released training/evaluation code embeds into existing COCO images. The video experiment is therefore an explicit framewise adapter at 512 x 512.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"
sys.path.insert(0, str(SCRIPTS))
from watermark_notebook_runtime import configured_path, require_commit, require_path

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

require = require_path

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in (
        "input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4",
        "effecterase.mp4", "e2fgvi-browser.mp4",
    ):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    result_json = {
        *run_root.rglob("summary.json"),
        *(attack / name for name in ("sample.json", "removal-pair.json", "effecterase.json", "e2fgvi.json")),
    }
    for path in sorted(path for path in result_json if path.is_file()):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


## Canonical one-clip comparison: all four inpainters

This is the canonical comparison for the minimum one-clip requirement. All four removers use the same frozen watermarked input and aligned binary mask. Every remover is presented in the same order, with the same output-level fields and the same video/JSON evidence. Method-specific resizing, padding, and mask dilation remain recorded in JSON rather than being silently normalized.

| Inpainting method | Status | Runtime (s) | Frames | Dimensions | FPS | Bytes | Evidence |
|---|---:|---:|---:|---:|---:|---:|---|
| ProPainter | complete | not recorded | 67 | 512 × 512 | 25.000 | 457,939 | [video](../../runs/remaining-watermarks/genptw/02b72b8d/attack/propainter.mp4) · [JSON](../../runs/remaining-watermarks/genptw/02b72b8d/attack/propainter.json) |
| DiffuEraser | complete | not recorded | 67 | 512 × 512 | 25.000 | 574,312 | [video](../../runs/remaining-watermarks/genptw/02b72b8d/attack/diffueraser.mp4) · [JSON](../../runs/remaining-watermarks/genptw/02b72b8d/attack/diffueraser.json) |
| EffectErase | complete | 101.351 | 67 | 512 × 512 | 25.000 | 321,963 | [video](../../runs/remaining-watermarks/genptw/02b72b8d/attack/effecterase.mp4) · [JSON](../../runs/remaining-watermarks/genptw/02b72b8d/attack/effecterase.json) |
| E2FGVI-HQ | complete | 21.855 | 67 | 512 × 512 | 25.000 | 667,506 | [playable video](../../runs/remaining-watermarks/genptw/02b72b8d/attack/e2fgvi-browser.mp4) · [evaluated MP4](../../runs/remaining-watermarks/genptw/02b72b8d/attack/e2fgvi.mp4) · [JSON](../../runs/remaining-watermarks/genptw/02b72b8d/attack/e2fgvi.json) |

Watermark method: **Genptw**. Runtime is shown only when the original run record captured it; `not recorded` is not replaced with an estimate. Watermark-specific localization or recovery metrics are downstream evaluations and are kept separate from this equal inpainting comparison. Where an evaluated codec is not browser-safe, the playable video is an H.264 viewing copy; the evaluated MP4 and exact JSON record remain unchanged.


In [ ]:
from pathlib import Path
import json
from IPython.display import JSON, Markdown, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/finalize_four_inpainter_notebooks.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
ATTACK = ROOT / 'runs/remaining-watermarks/genptw/02b72b8d/attack'
for name in ("input-watermarked.mp4", "watermarked.mp4", "gt-mask.mp4"):
    path = ATTACK / name
    if path.is_file():
        print(name)
        display(Video(str(path), embed=True, html_attributes="controls muted loop"))

REMOVERS = (
    ("ProPainter", "propainter"),
    ("DiffuEraser", "diffueraser"),
    ("EffectErase", "effecterase"),
    ("E2FGVI-HQ", "e2fgvi"),
)
for label, stem in REMOVERS:
    display(Markdown(f"### {label}"))
    evaluated_video = ATTACK / f"{stem}.mp4"
    browser_video = ATTACK / f"{stem}-browser.mp4"
    video = browser_video if browser_video.is_file() else evaluated_video
    metadata = ATTACK / f"{stem}.json"
    if not evaluated_video.is_file() or not video.is_file() or not metadata.is_file():
        raise FileNotFoundError(f"Missing {label} evidence in {ATTACK}")
    display(Video(str(video), embed=True, html_attributes="controls muted loop"))
    display(JSON(json.loads(metadata.read_text())))

for name in ("sample.json", "removal-pair.json"):
    metadata = ATTACK / name
    if metadata.is_file():
        print(name)
        display(JSON(json.loads(metadata.read_text())))


## Watermark evaluation results: all four inpainters

The downstream watermark evaluation is complete for this method. These are the exact recorded outputs for the same four canonical attack cases; they are not estimates.

| Inpainting method | Majority payload accuracy | Mean frame accuracy | Localization IoU |
| --- | ---: | ---: | ---: |
| ProPainter | 100.00% | 89.23% | 0.0013 |
| DiffuEraser | 100.00% | 97.46% | 0.0011 |
| EffectErase | 73.44% | 59.98% | 0.0013 |
| E2FGVI-HQ | 100.00% | 98.88% | 0.0012 |

**How to read this table:** Lower accuracy means less of the watermark remained recoverable after inpainting. Where IoU is reported, higher IoU means better overlap with the effective inpainted region.

Exact unrounded values and execution notes are in the [consolidated 44-case record](../../evidence/watermark-evaluation-44.json).


## Full-50 evaluation: key metrics

This section is generated from the machine-readable full-50 summary. It reports the required 50 samples and all four removers; incomplete GPU runs are shown as `pending`, never as zero.

Summary: `evidence/full50-watermark-evaluation-summary.json`  
SHA-256: `7c6cbbd52cd886b5237c2b060eedf4b641c277b510050f43b8541f63a8d8b347`

| Remover | Complete | Key metric aggregates (mean ± population SD; range) |
|---|---:|---|
| ProPainter | 50/50 | Video-majority bit accuracy: 0.9897 ± 0.0327; [0.8125, 1.0000]<br>Mean frame bit accuracy: 0.8645 ± 0.1181; [0.5852, 0.9971]<br>Spatial localization IoU: 0.0842 ± 0.1273; [0.0001, 0.5723] |
| DiffuEraser | 50/50 | Video-majority bit accuracy: 0.9997 ± 0.0022; [0.9844, 1.0000]<br>Mean frame bit accuracy: 0.9314 ± 0.0771; [0.7583, 1.0000]<br>Spatial localization IoU: 0.1289 ± 0.1617; [0.0000, 0.5811] |
| EffectErase | 50/50 | Video-majority bit accuracy: 0.8528 ± 0.1341; [0.5469, 1.0000]<br>Mean frame bit accuracy: 0.6480 ± 0.0916; [0.5073, 0.9201]<br>Spatial localization IoU: 0.0769 ± 0.1141; [0.0001, 0.5723] |
| E2FGVI-HQ | 50/50 | Video-majority bit accuracy: 0.9997 ± 0.0022; [0.9844, 1.0000]<br>Mean frame bit accuracy: 0.9412 ± 0.0728; [0.7474, 1.0000]<br>Spatial localization IoU: 0.1287 ± 0.1564; [0.0000, 0.5975] |

The next cell displays the complete per-sample table directly from the same summary and checks its hash before showing values.


In [ ]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

FULL50_METHOD = 'genptw'
FULL50_SUMMARY_SHA256 = '7c6cbbd52cd886b5237c2b060eedf4b641c277b510050f43b8541f63a8d8b347'
FULL50_SUMMARY_RELATIVE = Path("evidence/full50-watermark-evaluation-summary.json")

def find_full50_summary():
    starts = [Path.cwd(), Path.cwd().resolve()]
    for start in starts:
        for parent in (start, *start.parents):
            candidate = parent / FULL50_SUMMARY_RELATIVE
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f"Could not find {FULL50_SUMMARY_RELATIVE} from {Path.cwd()}")

summary_path = find_full50_summary()
actual_hash = hashlib.sha256(summary_path.read_bytes()).hexdigest()
if actual_hash != FULL50_SUMMARY_SHA256:
    raise RuntimeError(
        f"Full-50 summary changed (expected {FULL50_SUMMARY_SHA256}, found {actual_hash}). "
        "Run scripts/update_full50_watermark_notebooks.py before trusting this table."
    )
full50 = json.loads(summary_path.read_text())["methods"][FULL50_METHOD]
metric_labels = {item["key"]: item["label"] for item in full50["key_metrics"]}
headers = ["sample", "remover", "status", *metric_labels.values(), "evidence"]
rows = []
for sample in full50["samples"]:
    for remover, result in sample["removers"].items():
        values = [
            sample["sample_id"], remover, result["status"],
            *[result["metrics"].get(key) for key in metric_labels],
            result.get("evidence"),
        ]
        cells = "".join(
            f"<td>{html.escape('' if value is None else f'{value:.6g}' if isinstance(value, float) else str(value))}</td>"
            for value in values
        )
        rows.append(f"<tr>{cells}</tr>")
display(HTML(
    "<div style='max-height:560px;overflow:auto'><table>"
    + "<thead><tr>" + "".join(f"<th>{html.escape(value)}</th>" for value in headers) + "</tr></thead>"
    + "<tbody>" + "".join(rows) + "</tbody></table></div>"
))


In [ ]:
GENPTW = configured_path("GENPTW_ROOT", description="GenPTW checkout")
GENPTW_PYTHON = configured_path("GENPTW_PYTHON", default=GENPTW / ".venv/bin/python")
CHECKPOINT = GENPTW / "Checkpoint"
PRETRAINED_VAE = configured_path("GENPTW_PRETRAINED_VAE", description="Stable Diffusion 2 VAE")
CONVNEXT = configured_path("GENPTW_CONVNEXT", description="ConvNeXt checkpoint")
DIFFUERASER = configured_path("DIFFUERASER_ROOT", description="DiffuEraser checkout")
DIFFUERASER_PYTHON = configured_path("DIFFUERASER_PYTHON", default=DIFFUERASER / ".venv/bin/python")
GPU = os.environ.get("WATERMARK_GPU", "0")
MOSE_VIDEO = ROOT / "evidence/mose-task1/review50/media/02b72b8d/original.mp4"
MOSE_MASK = ROOT / "evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4"
RUN_ROOT = ROOT / "runs/remaining-watermarks/genptw/02b72b8d"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WATERMARKED_VIDEO = RUN_ROOT / "watermarked.mp4"
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"


In [ ]:
require_commit(GENPTW, "8c22fc04dcf9d896eaeabd034d09918588e82cb3")
for path, label in [
    (CHECKPOINT / "msg_decoder.pth", "GenPTW decoder"),
    (CHECKPOINT / "localizer.pth", "GenPTW localizer"),
    (CHECKPOINT / "diffusion_pytorch_model.safetensors", "GenPTW watermarked VAE"),
    (PRETRAINED_VAE, "Stable Diffusion 2 VAE"), (CONVNEXT, "ConvNeXt checkpoint"),
]: require(path, label)


In [ ]:
run([
    GENPTW_PYTHON, SCRIPTS / "genptw_video.py", "embed",
    "--source", GENPTW, "--checkpoint", CHECKPOINT,
    "--pretrained-vae", PRETRAINED_VAE, "--convnext-checkpoint", CONVNEXT,
    "--input-video", MOSE_VIDEO, "--input-mask", MOSE_MASK,
    "--output-video", WATERMARKED_VIDEO, "--output-mask", ALIGNED_MASK,
    "--seed", "2026",
])


## Run all four inpainting methods on the frozen input

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. Every remover below consumes that frozen input and mask.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "GenPTW",
        "--sample-id", "02b72b8d", "--sample-source", "MOSE fixed sample, GenPTW 512x512 image adapter",
        "--source-kind", "mose_post_hoc",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
EFFECTERASE = configured_path("EFFECTERASE_ROOT", description="EffectErase checkout")
EFFECTERASE_PYTHON = configured_path(
    "EFFECTERASE_PYTHON", default=EFFECTERASE / ".venv/bin/python"
)
E2FGVI = configured_path("E2FGVI_ROOT", description="E2FGVI checkout")
E2FGVI_CHECKPOINT = configured_path(
    "E2FGVI_CHECKPOINT", default=E2FGVI / "release_model/E2FGVI-HQ-CVPR22.pth"
)
E2FGVI_PYTHON = configured_path("E2FGVI_PYTHON", default=DIFFUERASER_PYTHON)

for path, label in (
    (EFFECTERASE_PYTHON, "EffectErase Python"),
    (E2FGVI_CHECKPOINT, "E2FGVI-HQ checkpoint"),
    (E2FGVI_PYTHON, "E2FGVI Python"),
):
    require(path, label)

if not (ATTACK_RUN / "effecterase.json").is_file():
    effect_env = os.environ.copy(); effect_env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        EFFECTERASE_PYTHON, SCRIPTS / "run_prepared_effecterase.py",
        "--source", EFFECTERASE, "--run", ATTACK_RUN, "--device", GPU,
    ], env=effect_env)
else:
    print("EffectErase output already exists:", ATTACK_RUN / "effecterase.json")

if not (ATTACK_RUN / "e2fgvi.json").is_file():
    run([
        E2FGVI_PYTHON, SCRIPTS / "run_prepared_e2fgvi.py",
        "--source", E2FGVI, "--checkpoint", E2FGVI_CHECKPOINT,
        "--run", ATTACK_RUN, "--python", E2FGVI_PYTHON, "--device", GPU,
    ])
else:
    print("E2FGVI-HQ output already exists:", ATTACK_RUN / "e2fgvi.json")


In [ ]:
run([
    GENPTW_PYTHON, SCRIPTS / "genptw_video.py", "detect",
    "--source", GENPTW, "--checkpoint", CHECKPOINT,
    "--pretrained-vae", PRETRAINED_VAE, "--convnext-checkpoint", CONVNEXT,
    "--gt-mask", ATTACK_RUN / "gt-mask.mp4", "--output", RUN_ROOT / "detection",
    "--case", f"original={ATTACK_RUN / 'input-watermarked.mp4'}",
    "--case", f"propainter={ATTACK_RUN / 'propainter.mp4'}",
    "--case", f"diffueraser={ATTACK_RUN / 'diffueraser.mp4'}",
    "--case", f"effecterase={ATTACK_RUN / 'effecterase.mp4'}",
    "--case", f"e2fgvi={ATTACK_RUN / 'e2fgvi.mp4'}",
])


In [ ]:
show_results(RUN_ROOT)
